In [2]:
import pandas as pd

In [6]:
from pathlib import Path

DATA_DIR = Path("/Users/narayanipemmaraju/Documents/MSDS/Spring 2026/ML/Project/Repo/StockTwit_WM/data/processed_week")

import pandas as pd
train = pd.read_parquet(DATA_DIR / "panel_train.parquet")
val   = pd.read_parquet(DATA_DIR / "panel_val.parquet")
test1 = pd.read_parquet(DATA_DIR / "panel_test1.parquet")
test2 = pd.read_parquet(DATA_DIR / "panel_test2.parquet")

print("Train:", train.shape, train['week'].min(), "→", train['week'].max())
print("Val:  ", val.shape,   val['week'].min(),   "→", val['week'].max())
print("Test1:", test1.shape, test1['week'].min(), "→", test1['week'].max())
print("Test2:", test2.shape, test2['week'].min(), "→", test2['week'].max())
print("\nColumns:", train.columns.tolist())

Train: (109737, 11) 2008-05-26 00:00:00 → 2018-12-31 00:00:00
Val:   (10400, 11) 2019-01-07 00:00:00 → 2019-12-30 00:00:00
Test1: (5200, 11) 2020-01-06 00:00:00 → 2020-06-29 00:00:00
Test2: (7800, 11) 2020-10-05 00:00:00 → 2021-06-28 00:00:00

Columns: ['symbol', 'week', 'msg_count', 'user_count', 'bullish_count', 'labeled_count', 'log_attention', 'bullish_rate', 'bearish_rate', 'unlabeled_rate', 'attn_growth']


In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys

DATA_DIR = Path("/Users/narayanipemmaraju/Documents/MSDS/Spring 2026/ML/Project/Repo/StockTwit_WM/data/processed_week")
sys.path.insert(0, str(DATA_DIR.parent.parent / "baselines"))

train = pd.read_parquet(DATA_DIR / "panel_train.parquet")
val   = pd.read_parquet(DATA_DIR / "panel_val.parquet")
test1 = pd.read_parquet(DATA_DIR / "panel_test1.parquet")
test2 = pd.read_parquet(DATA_DIR / "panel_test2.parquet")

# build fixed roster — top 100 tickers by total msg_count in training
K = 100
roster = (
    train.groupby('symbol')['msg_count']
    .sum()
    .sort_values(ascending=False)
    .head(K)
    .index.tolist()
)

# train/test split by week
train_weeks = sorted(train['week'].unique())
val_weeks   = sorted(val['week'].unique())
test1_weeks = sorted(test1['week'].unique())
test2_weeks = sorted(test2['week'].unique())

print(f"Roster: {len(roster)} tickers")
print(f"Top 5: {roster[:5]}")
print(f"Train weeks: {len(train_weeks)}")
print(f"Val weeks:   {len(val_weeks)}")
print(f"Test1 weeks: {len(test1_weeks)}")
print(f"Test2 weeks: {len(test2_weeks)}")

Roster: 100 tickers
Top 5: ['SPY', 'AAPL', 'AMD', 'TSLA', 'META']
Train weeks: 554
Val weeks:   52
Test1 weeks: 26
Test2 weeks: 39


In [8]:
from arima import PerTickerARIMA
from scipy.stats import spearmanr

arima = PerTickerARIMA(order=(2, 0, 1))
arima.fit(
    train, roster,
    log_attn_col='log_attention',
    week_col='week',
    symbol_col='symbol'
)

forecasts = arima.forecast(steps=13)

print("\n── Test1 (COVID) ──")
for h in [1, 4, 13]:
    if h > len(test1_weeks):
        print(f"Horizon {h}: not enough weeks")
        continue
    target_week = test1_weeks[h - 1]
    actual = (
        test1[test1['week'] == target_week]
        .groupby('symbol')['log_attention']
        .first()
    )
    preds, actuals = [], []
    for ticker in roster:
        if ticker in actual.index and ticker in forecasts:
            fc = forecasts[ticker]
            fc_val = fc.iloc[h-1] if hasattr(fc, 'iloc') else (fc[h-1] if fc.ndim == 1 else fc[h-1, 0])
            preds.append(float(fc_val))
            actuals.append(float(actual[ticker]))
    mse = np.mean((np.array(preds) - np.array(actuals)) ** 2)
    mae = np.mean(np.abs(np.array(preds) - np.array(actuals)))
    rho, _ = spearmanr(preds, actuals)
    print(f"Horizon {h:2d} → MSE: {mse:.4f} | MAE: {mae:.4f} | Spearman ρ: {rho:.4f}")

print("\n── Test2 (GME) ──")
for h in [1, 4, 13]:
    if h > len(test2_weeks):
        continue
    target_week = test2_weeks[h - 1]
    actual = (
        test2[test2['week'] == target_week]
        .groupby('symbol')['log_attention']
        .first()
    )
    preds, actuals = [], []
    for ticker in roster:
        if ticker in actual.index and ticker in forecasts:
            fc = forecasts[ticker]
            fc_val = fc.iloc[h-1] if hasattr(fc, 'iloc') else (fc[h-1] if fc.ndim == 1 else fc[h-1, 0])
            preds.append(float(fc_val))
            actuals.append(float(actual[ticker]))
    mse = np.mean((np.array(preds) - np.array(actuals)) ** 2)
    mae = np.mean(np.abs(np.array(preds) - np.array(actuals)))
    rho, _ = spearmanr(preds, actuals)
    print(f"Horizon {h:2d} → MSE: {mse:.4f} | MAE: {mae:.4f} | Spearman ρ: {rho:.4f}")

ARIMA fit: 100%|██████████| 100/100 [00:10<00:00,  9.44it/s]



── Test1 (COVID) ──
Horizon  1 → MSE: 0.5378 | MAE: 0.5386 | Spearman ρ: 0.5956
Horizon  4 → MSE: 0.5706 | MAE: 0.5537 | Spearman ρ: 0.5366
Horizon 13 → MSE: 1.1493 | MAE: 0.7938 | Spearman ρ: 0.4273

── Test2 (GME) ──
Horizon  1 → MSE: 2.9953 | MAE: 0.8538 | Spearman ρ: 0.4457
Horizon  4 → MSE: 2.0926 | MAE: 0.8238 | Spearman ρ: 0.4611
Horizon 13 → MSE: 3.8198 | MAE: 1.0578 | Spearman ρ: 0.3482


In [10]:
!pip install ReducedRankVAR

ERROR: Could not find a version that satisfies the requirement ReducedRankVAR (from versions: none)
ERROR: No matching distribution found for ReducedRankVAR


In [12]:
# find tickers that exist in all splits
common_tickers = (
    set(roster)
    & set(train['symbol'].unique())
    & set(test1['symbol'].unique())
    & set(test2['symbol'].unique())
)
roster_common = [t for t in roster if t in common_tickers]
print(f"Original roster: {len(roster)} → Common roster: {len(roster_common)}")

# rebuild train matrix with common roster
matrix_train = train[train['symbol'].isin(roster_common)].pivot_table(
    index='week', columns='symbol', values='log_attention', fill_value=0.0
).sort_index()[roster_common].values

# refit VAR on common roster
var2 = ReducedRankVAR(maxlags=4, rank=10)
var2.fit(train, roster_common, log_attn_col='log_attention', week_col='week', symbol_col='symbol')

last_obs = matrix_train[-var2.result.k_ar:]
forecasts_var = var_forecast_fixed(var2, last_obs, steps=13)

for split_name, split_df, split_weeks in [
    ("Test1 (COVID)", test1, test1_weeks),
    ("Test2 (GME)",   test2, test2_weeks)
]:
    print(f"\n── {split_name} ──")
    matrix_split = split_df[split_df['symbol'].isin(roster_common)].pivot_table(
        index='week', columns='symbol', values='log_attention', fill_value=0.0
    ).sort_index()[roster_common].values

    for h in [1, 4, 13]:
        if h > len(split_weeks):
            print(f"Horizon {h}: not enough weeks")
            continue
        predicted = forecasts_var[h - 1]
        actual    = matrix_split[h - 1]
        mse = np.mean((predicted - actual) ** 2)
        mae = np.mean(np.abs(predicted - actual))
        rho, _ = spearmanr(predicted, actual)
        print(f"Horizon {h:2d} → MSE: {mse:.4f} | MAE: {mae:.4f} | Spearman ρ: {rho:.4f}")

Original roster: 100 → Common roster: 65
[VAR] fitted lag=4, rank=10, K=65

── Test1 (COVID) ──
Horizon  1 → MSE: 47.5777 | MAE: 5.8185 | Spearman ρ: -0.0250
Horizon  4 → MSE: 54.0309 | MAE: 6.1969 | Spearman ρ: 0.1034
Horizon 13 → MSE: 52.7401 | MAE: 6.0326 | Spearman ρ: -0.0622

── Test2 (GME) ──
Horizon  1 → MSE: 39.3817 | MAE: 5.2962 | Spearman ρ: -0.0462
Horizon  4 → MSE: 51.3546 | MAE: 6.3470 | Spearman ρ: -0.1414
Horizon 13 → MSE: 41.4797 | MAE: 5.3113 | Spearman ρ: -0.0392


In [13]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from lstm import SharedLSTM, predict_lstm

FEATURES = ['log_attention', 'bullish_rate', 'bearish_rate', 'unlabeled_rate', 'attn_growth']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def build_tensor(panel_split, roster, features):
    matrices = []
    for feat in features:
        m = panel_split[panel_split['symbol'].isin(roster)].pivot_table(
            index='week', columns='symbol', values=feat, fill_value=0.0
        ).sort_index()[roster]
        matrices.append(m.values)
    return torch.tensor(np.stack(matrices, axis=-1).astype(np.float32))

# use common roster for consistency
train_tensor = build_tensor(train, roster_common, FEATURES)
test1_tensor  = build_tensor(test1, roster_common, FEATURES)
test2_tensor  = build_tensor(test2, roster_common, FEATURES)
print(f"Train tensor: {train_tensor.shape}")
print(f"Test1 tensor: {test1_tensor.shape}")
print(f"Test2 tensor: {test2_tensor.shape}")

# build sequences
SEQ_LEN = 8  # 8 week lookback (matches paper's window_k=8)
X, y = [], []
for i in range(len(train_tensor) - SEQ_LEN):
    X.append(train_tensor[i:i+SEQ_LEN])
    y.append(train_tensor[i+SEQ_LEN])

loader = DataLoader(
    TensorDataset(torch.stack(X), torch.stack(y)),
    batch_size=32, shuffle=True
)

# init model
K_common = len(roster_common)
lstm = SharedLSTM(n_tickers=K_common, feature_dim=5, hidden_dim=512, n_layers=2)
lstm = lstm.to(device)
optimizer = torch.optim.Adam(lstm.parameters(), lr=3e-4)

# train
for epoch in range(30):
    lstm.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        pred = lstm(X_batch)[:, -1]
        loss = nn.functional.mse_loss(pred, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(lstm.parameters(), 10.0)
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/30 — Loss: {total_loss/len(loader):.4f}")

# evaluate
context = train_tensor[-SEQ_LEN:]

for split_name, split_tensor, split_weeks in [
    ("Test1 (COVID)", test1_tensor, test1_weeks),
    ("Test2 (GME)",   test2_tensor, test2_weeks)
]:
    print(f"\n── {split_name} ──")
    preds_lstm = predict_lstm(lstm, context, steps=13, device=device)
    for h in [1, 4, 13]:
        if h > len(split_weeks):
            continue
        predicted = preds_lstm[h-1, :, 0]
        actual    = split_tensor[h-1, :, 0].numpy()
        mse = float(np.mean((predicted - actual) ** 2))
        mae = float(np.mean(np.abs(predicted - actual)))
        rho, _ = spearmanr(predicted, actual)
        print(f"Horizon {h:2d} → MSE: {mse:.4f} | MAE: {mae:.4f} | Spearman ρ: {float(rho):.4f}")

Using device: cpu
Train tensor: torch.Size([554, 65, 5])
Test1 tensor: torch.Size([26, 65, 5])
Test2 tensor: torch.Size([39, 65, 5])
Epoch 10/30 — Loss: 0.5087
Epoch 20/30 — Loss: 0.3670
Epoch 30/30 — Loss: 0.3227

── Test1 (COVID) ──
Horizon  1 → MSE: 6.0691 | MAE: 1.5839 | Spearman ρ: 0.5320
Horizon  4 → MSE: 9.3192 | MAE: 1.9305 | Spearman ρ: 0.4262
Horizon 13 → MSE: 16.0714 | MAE: 2.8917 | Spearman ρ: 0.3774

── Test2 (GME) ──
Horizon  1 → MSE: 21.5059 | MAE: 3.5365 | Spearman ρ: 0.3029
Horizon  4 → MSE: 16.6124 | MAE: 2.8949 | Spearman ρ: 0.3763
Horizon 13 → MSE: 22.9198 | MAE: 3.7589 | Spearman ρ: 0.4584


In [14]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from lstm import SharedLSTM, predict_lstm
from scipy.stats import spearmanr
import numpy as np

# ── device ──────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ── build tensors ────────────────────────────────────────────────
FEATURES = ['log_attention', 'bullish_rate', 'bearish_rate', 'unlabeled_rate', 'attn_growth']

def build_tensor(panel_split, roster, features):
    matrices = []
    for feat in features:
        m = panel_split[panel_split['symbol'].isin(roster)].pivot_table(
            index='week', columns='symbol', values=feat, fill_value=0.0
        ).sort_index()[roster]
        matrices.append(m.values)
    return torch.tensor(np.stack(matrices, axis=-1).astype(np.float32))

train_tensor = build_tensor(train, roster_common, FEATURES)
test1_tensor  = build_tensor(test1, roster_common, FEATURES)
test2_tensor  = build_tensor(test2, roster_common, FEATURES)
print(f"Train tensor: {train_tensor.shape}")
print(f"Test1 tensor: {test1_tensor.shape}")
print(f"Test2 tensor: {test2_tensor.shape}")

# ── build sequences ──────────────────────────────────────────────
SEQ_LEN = 8
X, y = [], []
for i in range(len(train_tensor) - SEQ_LEN):
    X.append(train_tensor[i:i+SEQ_LEN])
    y.append(train_tensor[i+SEQ_LEN])

loader = DataLoader(
    TensorDataset(torch.stack(X), torch.stack(y)),
    batch_size=32, shuffle=True
)

# ── init model ───────────────────────────────────────────────────
K_common = len(roster_common)
lstm = SharedLSTM(n_tickers=K_common, feature_dim=5, hidden_dim=512, n_layers=2)
lstm = lstm.to(device)
optimizer = torch.optim.Adam(lstm.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

# ── train 100 epochs ─────────────────────────────────────────────
for epoch in range(100):
    lstm.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        pred = lstm(X_batch)[:, -1]
        loss = nn.functional.mse_loss(pred, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(lstm.parameters(), 10.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/100 — Loss: {total_loss/len(loader):.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

# ── evaluate ─────────────────────────────────────────────────────
context = train_tensor[-SEQ_LEN:]

for split_name, split_tensor, split_weeks in [
    ("Test1 (COVID)", test1_tensor, test1_weeks),
    ("Test2 (GME)",   test2_tensor, test2_weeks)
]:
    print(f"\n── {split_name} ──")
    preds_lstm = predict_lstm(lstm, context, steps=13, device=device)
    for h in [1, 4, 13]:
        if h > len(split_weeks):
            continue
        predicted = preds_lstm[h-1, :, 0]
        actual    = split_tensor[h-1, :, 0].numpy()
        mse = float(np.mean((predicted - actual) ** 2))
        mae = float(np.mean(np.abs(predicted - actual)))
        rho, _ = spearmanr(predicted, actual)
        print(f"Horizon {h:2d} → MSE: {mse:.4f} | MAE: {mae:.4f} | Spearman ρ: {float(rho):.4f}")

# ── save model ───────────────────────────────────────────────────
torch.save(lstm.state_dict(), "lstm_baseline.pt")
print("\nSaved lstm_baseline.pt ✓")

Using device: mps
Train tensor: torch.Size([554, 65, 5])
Test1 tensor: torch.Size([26, 65, 5])
Test2 tensor: torch.Size([39, 65, 5])
Epoch 10/100 — Loss: 0.4335 | LR: 0.000293
Epoch 20/100 — Loss: 0.3298 | LR: 0.000271
Epoch 30/100 — Loss: 0.2843 | LR: 0.000238
Epoch 40/100 — Loss: 0.2645 | LR: 0.000196
Epoch 50/100 — Loss: 0.2582 | LR: 0.000150
Epoch 60/100 — Loss: 0.2376 | LR: 0.000104
Epoch 70/100 — Loss: 0.2341 | LR: 0.000062
Epoch 80/100 — Loss: 0.2269 | LR: 0.000029
Epoch 90/100 — Loss: 0.2311 | LR: 0.000007
Epoch 100/100 — Loss: 0.2223 | LR: 0.000000

── Test1 (COVID) ──
Horizon  1 → MSE: 5.9190 | MAE: 1.8160 | Spearman ρ: 0.5139
Horizon  4 → MSE: 7.6435 | MAE: 2.0707 | Spearman ρ: 0.5380
Horizon 13 → MSE: 11.2785 | MAE: 2.6092 | Spearman ρ: 0.5535

── Test2 (GME) ──
Horizon  1 → MSE: 16.1714 | MAE: 3.3477 | Spearman ρ: 0.3245
Horizon  4 → MSE: 12.3459 | MAE: 2.8106 | Spearman ρ: 0.4808
Horizon 13 → MSE: 17.2411 | MAE: 3.4343 | Spearman ρ: 0.4190

Saved lstm_baseline.pt ✓
